<a href="https://colab.research.google.com/github/daniivelascoo/ifp-programacion-ia/blob/main/Lab_2_1_Churn_Methodology_Student.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📉 Laboratorio 2.1: Proyecto CHURN (Metodología ML)
**Núcleo Formativo 2 - Machine Learning**

---
### 📜 Contexto del Negocio
Una empresa de telecomunicaciones está preocupada por la fuga de clientes ("Churn").
Quieren una Inteligencia Artificial que analice los datos de facturación y antigüedad para predecir **quién se va a ir** antes de que ocurra, para poder ofrecerles descuentos.

### 🎯 Tu Misión
No vamos a centrarnos en limpiar datos hoy (eso ya te lo damos hecho).
Tu misión es aplicar la **Metodología Científica** para entrenar un modelo:
1.  Definir qué aprendemos (**X**) y qué predecimos (**y**).
2.  Dividir la realidad en dos universos (**Train** y **Test**) para no hacer trampas.
3.  Entrenar un **Árbol de Decisión**.
4.  Evaluar su rendimiento real.

---

In [1]:
# --- ⚙️ 0. CONFIGURACIÓN Y DATOS (NO TOCAR) ---
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# URL Estable del dataset Telco Customer Churn (IBM)
url = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"

try:
    print("📡 Descargando datos de la compañía...")
    df = pd.read_csv(url)

    # --- PRE-PROCESAMIENTO AUTOMÁTICO ---
    # Limpiamos los datos para que tú solo te preocupes del ML

    # 1. Target a números: Yes=1 (Se va), No=0 (Se queda)
    df['Churn'] = df['Churn'].apply(lambda x: 1 if x == 'Yes' else 0)

    # 2. Convertir TotalCharges a numérico (hay espacios vacíos que dan error)
    df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
    df = df.dropna() # Borrar los pocos nulos

    # 3. Seleccionar solo variables numéricas relevantes para este ejercicio
    cols = ['tenure', 'MonthlyCharges', 'TotalCharges', 'Churn']
    df = df[cols]

    print(f"✅ Datos listos. Dimensiones: {df.shape}")
    print("Variables: 'tenure' (meses de antigüedad), 'MonthlyCharges' (factura mes), 'TotalCharges' (total pagado).")
    display(df.head())

except Exception as e:
    print(f"❌ Error de carga: {e}")

📡 Descargando datos de la compañía...
✅ Datos listos. Dimensiones: (7032, 4)
Variables: 'tenure' (meses de antigüedad), 'MonthlyCharges' (factura mes), 'TotalCharges' (total pagado).


,tenure,MonthlyCharges,TotalCharges,Churn
0,1,29.85,29.85,0
1,34,56.95,1889.50,0
2,2,53.85,108.15,1
3,45,42.30,1840.75,0
4,2,70.70,151.65,1


---
## 🧠 FASE 1: Definición del Problema (X e y)

*   **Features (X):** ¿Qué datos usará la máquina para estudiar? (Antigüedad y dinero).
*   **Target (y):** ¿Qué queremos adivinar? (Si se va o no: `Churn`).

In [3]:
# 1. Crea X con las columnas 'tenure', 'MonthlyCharges', 'TotalCharges'
# Pista: Usa doble corchete df[[...]]
X = df[['tenure', 'MonthlyCharges', 'TotalCharges']]

# 2. Crea y con la columna 'Churn'
y = df['Churn']

print(f"X (Preguntas): {X.shape}")
print(f"y (Respuestas): {y.shape}")

X (Preguntas): (7032, 3)
y (Respuestas): (7032,)


---
## ✂️ FASE 2: La Regla de Oro (Train / Test Split)

Nunca evalúes un modelo con los datos que usó para aprender. Sería como enseñar las preguntas del examen antes de hacerlo.

**Misión:**
*   Usa `train_test_split` para separar el 20% de los datos para el examen final.
*   Fija `random_state=42` para que todos tengamos el mismo resultado.

In [5]:
from sklearn.model_selection import train_test_split

# Rellena los huecos
X_train, X_test, y_train, y_test = train_test_split(
    X,    # Tus datos X
    y,    # Tus datos y
    test_size=0.2,  # Porcentaje para test (0.2)
    random_state=42
)

print("Datos separados correctamente.")
print(f"Estudiaremos con {len(X_train)} clientes.")
print(f"Haremos el examen con {len(X_test)} clientes.")

Datos separados correctamente.
Estudiaremos con 5625 clientes.
Haremos el examen con 1407 clientes.


---
## 🌳 FASE 3: Entrenamiento (Fit)

Vamos a usar un **Árbol de Decisión** (`DecisionTreeClassifier`). Es un modelo que aprende creando reglas tipo "Si paga más de 50€ y lleva poco tiempo -> Se va".

**Misión:**
1.  Crea el modelo (instancia).
2.  Entrénalo (`.fit`) usando SOLO los datos de **TRAIN**.

In [6]:
from sklearn.tree import DecisionTreeClassifier

# 1. Crear el cerebro (fija random_state=42 también aquí)
modelo = DecisionTreeClassifier(random_state=42, max_depth=5)
# (Ponemos max_depth=5 para evitar que el árbol se vuelva loco memorizando)

# 2. Entrenar (Estudiar)
print("🧠 Entrenando modelo...")
modelo.fit(X_train, y_train) # ¡Usa X_train e y_train!

print("¡Entrenamiento finalizado!")

🧠 Entrenando modelo...
¡Entrenamiento finalizado!


---
## 📝 FASE 4: Evaluación (Predict & Metrics)

Ahora pasamos el examen. Vamos a pedirle al modelo que prediga qué harán los clientes del grupo **TEST** (que nunca ha visto).

**Misión:**
1.  Genera predicciones sobre `X_test`.
2.  Calcula el `accuracy_score` comparando tus predicciones con la realidad (`y_test`).
3.  Muestra la Matriz de Confusión.

In [7]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# 1. Predecir (El Examen)
predicciones = modelo.predict(X_test) # ¡Usa X_test!

# 2. Calcular nota (Accuracy)
nota = accuracy_score(y_test, predicciones) # Compara y_test con predicciones

print(f"Precisión del modelo: {nota:.1%}")

# 3. Matriz de Confusión
print("\n--- Matriz de Confusión ---")
# Muestra cuántos acertó y cuántos falló
print(confusion_matrix(y_test, predicciones))

Precisión del modelo: 78.2%

--- Matriz de Confusión ---
[[945  88]
 [219 155]]


### 🧐 REFLEXIÓN DE NEGOCIO (Obligatorio)
*Haz doble clic para editar.*

Si el modelo tiene una precisión del 78% (aprox), ¿crees que es perfecto? ¿Qué es peor para la empresa: pensar que un cliente se va y que se quede (Falso Positivo), o pensar que se queda y que se vaya sin que le ofrezcamos nada (Falso Negativo)?

**Respuesta:**

---
## 🏁 VALIDACIÓN FINAL
Ejecuta la celda de abajo para comprobar tu trabajo.

In [8]:
# --- 🤖 CÓDIGO DE VALIDACIÓN (NO MODIFICAR) ---
def validar_churn():
    print("🚀 AUDITANDO METODOLOGÍA ML...\n")
    puntos = 0
    errores = []

    v_X_train = globals().get('X_train')
    v_X_test = globals().get('X_test')
    v_modelo = globals().get('modelo')

    # 1. VALIDACIÓN SPLIT
    if v_X_train is not None and v_X_test is not None:
        total = len(v_X_train) + len(v_X_test)
        ratio = len(v_X_test) / total
        if 0.19 < ratio < 0.21:
            print("✅ [FASE 2] Train/Test Split: CORRECTO (20% Test).")
            puntos += 3.5

            # Verificar Data Leakage (Intersección de índices)
            if len(set(v_X_train.index).intersection(set(v_X_test.index))) == 0:
                print("   -> Integridad de datos: PERFECTA (Sin fugas).")
            else:
                errores.append("❌ ALERTA CRÍTICA: Hay datos repetidos en Train y Test (Data Leakage).")
        else:
            errores.append(f"❌ El tamaño del test no es correcto ({ratio:.2%}). Revisa test_size.")
    else:
        errores.append("❌ No has definido X_train / X_test.")

    # 2. VALIDACIÓN MODELO
    if v_modelo is not None:
        # Verificar si está entrenado
        if hasattr(v_modelo, "tree_"):
            print("✅ [FASE 3] Modelo Entrenado: CORRECTO.")
            puntos += 3.5
        else:
            errores.append("❌ El modelo existe pero no ha sido entrenado (.fit).")
    else:
        errores.append("❌ No existe la variable 'modelo'.")

    # 3. VALIDACIÓN PREDICCIÓN
    v_pred = globals().get('predicciones')
    if v_pred is not None:
        if len(v_pred) == len(v_X_test):
            print("✅ [FASE 4] Evaluación: CORRECTA.")
            puntos += 3
        else:
            errores.append("❌ Las predicciones no coinciden con el tamaño del test.")
    else:
        errores.append("❌ No has generado la variable 'predicciones'.")

    # REPORTE
    print("\n" + "="*50)
    if puntos == 10:
        import hashlib
        code = hashlib.md5(str(len(v_X_train)).encode()).hexdigest()[:8].upper()
        print(f"🎉 ¡METODOLOGÍA APROBADA! Estás listo para el Hito 2.")
        print(f"🔐 CÓDIGO DE VALIDACIÓN: CHURN-{code}")
    else:
        print("⚠️ REVISA LOS ERRORES:")
        for e in errores: print(f"   - {e}")
    print("="*50)

validar_churn()

🚀 AUDITANDO METODOLOGÍA ML...

✅ [FASE 2] Train/Test Split: CORRECTO (20% Test).
   -> Integridad de datos: PERFECTA (Sin fugas).
✅ [FASE 3] Modelo Entrenado: CORRECTO.
✅ [FASE 4] Evaluación: CORRECTA.

🎉 ¡METODOLOGÍA APROBADA! Estás listo para el Hito 2.
🔐 CÓDIGO DE VALIDACIÓN: CHURN-DA94BE6D
